<a href="https://colab.research.google.com/github/Mirajul100/Machine-Learning/blob/master/Emotions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [335]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [336]:
import re

In [337]:
df = pd.read_csv('/content/train.txt' , sep=';' , header=None , names=['text' , 'emotion'])

In [338]:
df.head(5)

,text,emotion
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger


In [339]:
df.shape

(16000, 2)

In [340]:
df.isnull().sum()

,0
text,0
emotion,0


In [341]:
df.duplicated().sum()

np.int64(1)

In [342]:
from sklearn.preprocessing import LabelEncoder

In [343]:
encode = LabelEncoder()

In [344]:
encode.fit(df['emotion'])

LabelEncoder()

In [345]:
df['emotion'] = encode.transform(df['emotion'])

In [346]:
df.head(5)

,text,emotion
0,i didnt feel humiliated,4
1,i can go from feeling so hopeless to so damned...,4
2,im grabbing a minute to post i feel greedy wrong,0
3,i am ever feeling nostalgic about the fireplac...,3
4,i am feeling grouchy,0


# Convert text into lowercase

In [347]:
df['text'] = df['text'].str.lower()

In [348]:
df.head(5)

,text,emotion
0,i didnt feel humiliated,4
1,i can go from feeling so hopeless to so damned...,4
2,im grabbing a minute to post i feel greedy wrong,0
3,i am ever feeling nostalgic about the fireplac...,3
4,i am feeling grouchy,0


# Remove punctuations

In [349]:
df['text'] = df['text'].str.replace('[^\w\s]' , '' , regex=True)

# Remove digits

In [350]:
def remove_digits(text):
  new = ''
  for i in text:
    if not i.isdigit():
      new = new + i
  return new

In [351]:
df['text'] = df['text'].apply(remove_digits)

# Remove emojis

In [352]:
def remove_emoji(text):
  new = ''
  for i in text:
    if i.isascii():
      new = new + i
  return new

In [353]:
df['text'] = df['text'].apply(remove_emoji)

# Remove link

In [354]:
import re

In [355]:
def remove_link(text):
  return re.sub(r'http\S+' , '' , text)

In [356]:
df['text'] = df['text'].apply(remove_link)

# Remove html tags

In [357]:
def remove_tags(text):
 text = re.sub(r'<.*?>', '', text)
 return text

In [358]:
df['text'] = df['text'].apply(remove_tags)

# Remove stop words

In [359]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

In [360]:
nltk.download('stopwords')
nltk.download('punkt_tab')
nltk.download('punkt')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [361]:
stop_words = set(stopwords.words('english'))
len(stop_words)

198

In [362]:
def remove_stopwords(text):
 word = word_tokenize(text)
 new = []
 for i in word:
   if i not in stop_words:
     new.append(i)
 return ' '.join(new)

In [363]:
df['text'] = df['text'].apply(remove_stopwords)

In [364]:
df.head(5)

,text,emotion
0,didnt feel humiliated,4
1,go feeling hopeless damned hopeful around some...,4
2,im grabbing minute post feel greedy wrong,0
3,ever feeling nostalgic fireplace know still pr...,3
4,feeling grouchy,0


### Distribution of Emotion Labels

In [ ]:
emotion_counts = df['emotion'].value_counts().sort_index()
emotion_labels = encode.inverse_transform(emotion_counts.index)

plt.figure(figsize=(10, 6))
sns.barplot(x=emotion_labels, y=emotion_counts.values, palette='viridis')
plt.title('Distribution of Emotion Labels')
plt.xlabel('Emotion')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [365]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.feature_extraction.text import TfidfVectorizer

In [366]:
X = df['text']
y = df['emotion']

In [367]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

In [368]:
bow_vectorizer = CountVectorizer()
X_train_bow = bow_vectorizer.fit_transform(X_train)
X_test_bow = bow_vectorizer.transform(X_test)

In [369]:
model_nb = MultinomialNB()
model_nb.fit(X_train_bow, y_train)

MultinomialNB()

In [371]:
y_pre_bow = model_nb.predict(X_test_bow)
acc = accuracy_score(y_test , y_pre_bow)
print(acc)

0.7649621212121213


In [372]:
tfidf_vectorizer = TfidfVectorizer()
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

In [373]:
model_nb = MultinomialNB()
model_nb.fit(X_train_tfidf, y_train)

MultinomialNB()

In [375]:
y_pre_tfidf = model_nb.predict(X_test_tfidf)
acc = accuracy_score(y_test , y_pre_tfidf)
print(acc)

0.6609848484848485


In [376]:
model_lo = LogisticRegression()
model_lo.fit(X_train_bow, y_train)

LogisticRegression()

In [377]:
y_pre_bow = model_lo.predict(X_test_bow)
acc = accuracy_score(y_test , y_pre_bow)
print(acc)

0.887689393939394


### Save the best performing model and its vectorizer

In [385]:
import pickle

with open('logistic_regression_bow_model.pkl', 'wb') as f:
    pickle.dump(model_lo, f)
with open('count_vectorizer.pkl', 'wb') as f:
    pickle.dump(bow_vectorizer, f)

print("Logistic Regression model and CountVectorizer saved successfully.")

Logistic Regression model and CountVectorizer saved successfully.


In [378]:
model_lo = LogisticRegression()
model_lo.fit(X_train_tfidf, y_train)

LogisticRegression()

In [379]:
y_pre_tfidf = model_lo.predict(X_test_tfidf)
acc = accuracy_score(y_test , y_pre_tfidf)
print(acc)

0.8473484848484848
